# Step 1 — Get and understand AMLBench (Yellow Bank)

**What this notebook does (run cells top to bottom):**
1. Installs the tools
2. Connects your Google Drive (to save the reports)
3. Downloads *Yello Bank.zip* (9.4 GB) from Hugging Face **into Colab, not your laptop**
4. Unzips it and lists every file
5. Summarises the answer key (`labels.json`): laundering customers, typologies, signal columns
6. Builds a **data dictionary** (every table, its columns, row counts, examples)
7. Gives a first look at transactions and customers
8. Saves all reports to your Google Drive folder `AML_project/step1_reports`

Runtime: about 20–40 minutes (mostly the download). Use **Runtime → Change runtime type → CPU (High-RAM if available)**. No GPU needed.

In [ ]:
# 1) Install tools
!pip install -q duckdb huggingface_hub pyarrow

In [ ]:
# 2) Connect Google Drive so reports are saved permanently
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = '/content/drive/MyDrive/AML_project/step1_reports'
os.makedirs(OUT, exist_ok=True)
print('Reports will be saved to:', OUT)

In [ ]:
# 3) Download Yellow Bank (9.4 GB) — takes ~10-25 minutes
from huggingface_hub import hf_hub_download
zip_path = hf_hub_download(repo_id='DVK2026/AMLBench', repo_type='dataset',
                           filename='Yello Bank.zip', local_dir='/content/amlbench')
print('Downloaded to:', zip_path)
!df -h /content | tail -1

In [ ]:
# 4) Unzip, then delete the zip to free disk space
!mkdir -p /content/amlbench/yellow
!unzip -q "/content/amlbench/Yello Bank.zip" -d /content/amlbench/yellow
!rm "/content/amlbench/Yello Bank.zip"
!df -h /content | tail -1

In [ ]:
# 5) List every file with its size
import os
ROOT = '/content/amlbench/yellow'
rows = []
for dp, dn, fn in os.walk(ROOT):
    for f in fn:
        p = os.path.join(dp, f)
        rows.append((os.path.relpath(p, ROOT), round(os.path.getsize(p)/1e6, 1)))
rows.sort()
for r in rows[:300]:
    print(f'{r[1]:>10} MB   {r[0]}')
print('Total files:', len(rows))
import pandas as pd
pd.DataFrame(rows, columns=['file', 'size_MB']).to_csv(f'{OUT}/file_list.csv', index=False)

In [ ]:
# 6) Find labels.json and all Parquet tables automatically
import glob, json, collections
labels_files = glob.glob(f'{ROOT}/**/labels.json', recursive=True)
print('labels.json found at:', labels_files)

pq = glob.glob(f'{ROOT}/**/*.parquet', recursive=True)
tables = collections.defaultdict(list)
for p in pq:
    stem = os.path.splitext(os.path.basename(p))[0]
    # files like part-0.parquet live inside a folder named after the table
    name = os.path.basename(os.path.dirname(p)) if stem.startswith('part') else stem
    tables[name.lower()].append(p)
print(len(tables), 'tables found:')
for t in sorted(tables): print('  ', t, f'({len(tables[t])} file(s))')

In [ ]:
# 7) The answer key: labels.json
with open(labels_files[0]) as f:
    labels = json.load(f)
node_labels = labels.get('node_labels', {})
scen = labels.get('scenarios', [])
n_aml = sum(1 for v in node_labels.values() if v == 1)
print('Customers with a label :', len(node_labels))
print('Laundering customers   :', n_aml)
print('Scenarios              :', len(scen))

typ = pd.Series([s['typology'] for s in scen]).value_counts()
print('\nScenarios per typology:'); print(typ)
typ.to_csv(f'{OUT}/typology_counts.csv')

sig = pd.Series([c for s in scen for c in s.get('signal_columns', [])]).value_counts()
print('\nMost common signal columns (evidence fields):'); print(sig.head(30))
sig.to_csv(f'{OUT}/signal_column_counts.csv')

print('\nExample scenario:'); print(json.dumps(scen[0], indent=2)[:1500])

pd.DataFrame([{'scenario_id': s['scenario_id'], 'typology': s['typology'],
               'n_customers': len(s.get('customer_ids', [])), 'n_accounts': len(s.get('account_ids', [])),
               'date_from': s.get('date_from'), 'date_to': s.get('date_to'),
               'signal_columns': '; '.join(s.get('signal_columns', []))} for s in scen]
            ).to_csv(f'{OUT}/scenarios.csv', index=False)

In [ ]:
# 8) Data dictionary: every table -> columns, types, row count, example values
import duckdb
con = duckdb.connect()
def src(t):
    files = tables[t]
    return "read_parquet([" + ", ".join(f"'{f}'" for f in files) + "])"

dd = []
for t in sorted(tables):
    try:
        n = con.execute(f'SELECT COUNT(*) FROM {src(t)}').fetchone()[0]
        sample = con.execute(f'SELECT * FROM {src(t)} LIMIT 3').df()
        for col in sample.columns:
            dd.append({'table': t, 'rows': n, 'column': col, 'dtype': str(sample[col].dtype),
                       'example': str(sample[col].iloc[0])[:80] if len(sample) else ''})
        print(f'{t:40s} {n:>12,} rows  {len(sample.columns)} columns')
    except Exception as e:
        print('Could not read', t, e)
dd = pd.DataFrame(dd)
dd.to_csv(f'{OUT}/data_dictionary.csv', index=False)
print('\nSaved data_dictionary.csv — open it and write the meaning of each column.')

In [ ]:
# 9) First look: customers, KYC and business profiles
for t in ['customer_master', 'customer_business', 'customer_kyc', 'customer_financial_profile', 'customer_risk_profile']:
    if t in tables:
        print('\n=====', t); display(con.execute(f'SELECT * FROM {src(t)} LIMIT 5').df())

In [ ]:
# 10) First look: transactions (date range, amount, channels, descriptions)
t = 'transaction_base'
cols = con.execute(f'SELECT * FROM {src(t)} LIMIT 1').df().columns.tolist()
print('Columns:', cols)
time_cols = [c for c in cols if 'time' in c.lower() or 'date' in c.lower()]
amt_cols  = [c for c in cols if 'amount' in c.lower()]
txt_cols  = [c for c in cols if 'desc' in c.lower() or 'narr' in c.lower() or 'memo' in c.lower()]
cat_cols  = [c for c in cols if c.lower() in ('channel', 'transaction_type', 'txn_type', 'type')]
print('time:', time_cols, '| amount:', amt_cols, '| text:', txt_cols, '| category:', cat_cols)
if time_cols:
    display(con.execute(f'SELECT MIN({time_cols[0]}) AS first, MAX({time_cols[0]}) AS last FROM {src(t)}').df())
if amt_cols:
    display(con.execute(f'SELECT MIN({amt_cols[0]}), AVG({amt_cols[0]}), MAX({amt_cols[0]}) FROM {src(t)}').df())
for c in cat_cols + txt_cols:
    print('\nTop values of', c)
    display(con.execute(f'SELECT {c}, COUNT(*) n FROM {src(t)} GROUP BY 1 ORDER BY 2 DESC LIMIT 15').df())

In [ ]:
# 11) How many laundering customers are actually in this bank?
cm_cols = con.execute(f"SELECT * FROM {src('customer_master')} LIMIT 1").df().columns.tolist()
id_col = [c for c in cm_cols if 'customer' in c.lower() and 'id' in c.lower()][0]
ids = set(con.execute(f"SELECT DISTINCT {id_col} FROM {src('customer_master')}").df()[id_col].astype(str))
in_bank = {k: v for k, v in node_labels.items() if k in ids}
print('Customers in Yellow Bank      :', len(ids))
print('...with a label               :', len(in_bank))
print('...laundering (label 1)       :', sum(v == 1 for v in in_bank.values()))
summary = {'customers': len(ids), 'labelled': len(in_bank), 'laundering': sum(v == 1 for v in in_bank.values()),
           'scenarios': len(scen), 'tables': len(tables)}
pd.Series(summary).to_csv(f'{OUT}/summary.csv')
print('\nDone. All reports are in Google Drive:', OUT)

## What to send back
Download these files from Google Drive → `AML_project/step1_reports` and share them (or put them in the `data/reports` folder on your laptop):
`summary.csv`, `typology_counts.csv`, `signal_column_counts.csv`, `scenarios.csv`, `data_dictionary.csv`, `file_list.csv`.

They are small text files. With them we can plan the features, the alert rules and the benchmark exactly.

**Note:** Colab deletes `/content` when the session ends. Next time, either re-run cells 1–4, or (better) copy the parquet folder to Drive once:
`!cp -r /content/amlbench/yellow /content/drive/MyDrive/AML_project/` (needs ~10–15 GB free in Drive).